# TASK 4: Visualizing Fire Scars through False Color
Github Repository Link: https://github.com/nberry18/eds220-hwk4.git

In February 2021, a severe winter storm swept across the U.S. central plains, setting national records for low temperatures. Texas was especially affected, with widespread electricity and water outages. This event ranks among the most severe winter weather incidents in history, resulting in loss of life and long-term economic consequences. In the aftermath, political debate and blame can obscure the facts.

# Data Source:

https://data.lacounty.gov/maps/ad51845ea5fb4eb483bc2a7c38b2370c/about 
- Palisades Fire perimeter
- Eaton Fire

# About the data

In this task you will use two datasets:
First dataset: oil spills data

The first dataset contains information about spilling incidents in the state of New York. Make sure you study the data portal and understand what each row represents before starting to code. You will find extra information and metadata for this dataset in the NYSDEC_SpillIncidents_Overview.pdf and NYSDEC_SpillIncidents_DataDictionary.pdf documents available at the portal.
Second dataset: TIGER shapefile

The second dataset is the 2026 Counties (and equivalent) TIGER shapefile (file tl_2026_us_county.zip) from the Census Bureau. You can check the metadata for all the TIGER shapefiles here. 


# 1. Import Libraries 

In [1]:
#import packages
import os # Constructs operating-system-independent file paths 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # Controls the figure canvas, styling, axes, and image 
import geopandas as gpd # Reads spatial vector formats (Shapefiles, GeoJSON, GeoPackage) 
import xarray as xr # pandas for multi-dimensional arrays
import rioxarray #  objects for spatial tasks for rasters

# 2. Import Data Fire Perimeter Data

In [2]:
# Read in Eaton and Palisades fires data from LA County
eaton = gpd.read_file(os.path.join(
                      'data',
                      'Eaton_Perimeter_20250121',
                      'Eaton_Perimeter_20250121.shp'))

palisades = gpd.read_file(os.path.join(
                         'data',
                         'Palisades_Perimieter_20250121',
                         'Palisades_Perimieter_20250121.shp'))

# 3. Eaton and Palisades fire perimeter data exploration

In [3]:
# Check # of polygons and CRS
print(f"Palisades: {palisades.shape[0]} polygon(s), CRS = EPSG:{palisades.crs.to_epsg()}") # print statement
print(f"Eaton:     {eaton.shape[0]} polygon(s), CRS = EPSG:{eaton.crs.to_epsg()}")

Palisades: 1 polygon(s), CRS = EPSG:4269
Eaton:     1 polygon(s), CRS = EPSG:4269


In [4]:
print(eaton.head()) 
print(eaton.info)
print(eaton.crs)

   OBJECTID  YEAR_ STATE AGENCY UNIT_ID FIRE_NAME            type   INC_NUM  \
0         1   2025    CA    CDF     LAC     EATON  Heat Perimeter  00009087   

                                  IRWINID            ALARM_DATE  ...  \
0  {72660ADC-B5EF-4D96-A33F-B4EA3740A4E3}  2025-01-08T00:00:00Z  ...   

  GIS_ACRES  COMMENTS  COMPLEX_NA  COMPLEX_ID  FIRE_NUM  \
0  14056.26      None        None        None      None   

                               GlobalID    DECADES Shape__Are Shape__Len  \
0  000a99c2-0fef-4960-a3c4-f2342a0cd291  2020-2025   0.005564   0.859026   

                                            geometry  
0  MULTIPOLYGON (((-118.10093 34.16680, -118.1008...  

[1 rows x 24 columns]
<bound method DataFrame.info of    OBJECTID  YEAR_ STATE AGENCY UNIT_ID FIRE_NAME            type   INC_NUM  \
0         1   2025    CA    CDF     LAC     EATON  Heat Perimeter  00009087   

                                  IRWINID            ALARM_DATE  ...  \
0  {72660ADC-B5EF-4D96-A33F-

The Eaton fire dataset has a CRS of 4269 and is made up of 1 polygon

In [5]:
# Exploring Palisades data
print(palisades.head()) # looking at polygon
print(palisades.info)
print(palisades.crs)

   OBJECTID  YEAR_ STATE AGENCY UNIT_ID  FIRE_NAME            type   INC_NUM  \
0         1   2025    CA    CDF     LDF  PALISADES  Heat Perimeter  00000738   

                                  IRWINID            ALARM_DATE  ...  \
0  {A7EA5D21-F882-44B8-BF64-44AB11059DC1}  2025-01-07T00:00:00Z  ...   

  GIS_ACRES  COMMENTS  COMPLEX_NA  COMPLEX_ID  FIRE_NUM  \
0  23448.88      None        None        None      None   

                               GlobalID    DECADES Shape__Are Shape__Len  \
0  8aea8178-a9d8-4c10-a7fd-d1371091b114  2020-2025   0.009268   1.157614   

                                            geometry  
0  MULTIPOLYGON (((-118.51974 34.02996, -118.5198...  

[1 rows x 24 columns]
<bound method DataFrame.info of    OBJECTID  YEAR_ STATE AGENCY UNIT_ID  FIRE_NAME            type   INC_NUM  \
0         1   2025    CA    CDF     LDF  PALISADES  Heat Perimeter  00000738   

                                  IRWINID            ALARM_DATE  ...  \
0  {A7EA5D21-F882-44B8-B

# 4. NetCDF data import and exploration

In [ ]:
landsat = xr.open_dataset('data/landsat8-2025-02-23-palisades-eaton.nc')

In [ ]:
landsat.head()

Explore the data and write a brief summary of the information you obtained from the preliminary exploration. Your summary should include a description of the data’s variables, dimensions, and coordinates.

In [ ]:
landsat.info

In [ ]:
print("The CRS of this data is:", landsat.rio.crs)

In [ ]:
landsat.spatial_ref.crs_wkt

In [ ]:
landsat.rio.write_crs("EPSG:32611", inplace=True)

In [ ]:
landsat.rio.crs


    Without creating any new variables:

    select the red, green, and blue variables (in that order) of the xarray.Dataset holding the Landsat data,
    convert it to an xarray.DataArray using the to_dataarray() method, and then
    use .plot.imshow() to create an RGB image with the data. There will be a warning; that’s ok.


In [ ]:
landsat[['red','green','blue']].to_dataarray().plot.imshow()

In [ ]:
landsat[['red','green','blue']].to_dataarray().plot.imshow(robust=True)

In [ ]:
bands = ['red','green','blue']

for band in bands:
    na_total = int(np.isnan(landsat[band]).sum())
    print(f"The {band} band has {na_total} NA values.")


In [ ]:
landsat = landsat.fillna(value=0) 

In [ ]:
landsat[['red','green','blue']].to_dataarray().plot.imshow(robust=True)

In a markdown cell write a brief explanation comparing the outputs for parts (a) and (e).

# 6. False color image

In [ ]:
landsat[["swir22", "nir08", "red"]].to_dataarray().plot.imshow(robust=True)

# 7. Map

Create a map showing the shortwave infrared/near-infrared/red false color image together with both fire perimeters. Customize it appropriately including, at least, an informative title and legend. You may also want to include text on the map to identify which fire is which.

Write a figure description for the map including a brief explanation of how false color imagery is being used.

In [ ]:
eaton = eaton.to_crs(landsat.rio.crs)
palisades = palisades.to_crs(landsat.rio.crs)

In [ ]:
# Finding max coordinates of fire polygons 
eaton_max = eaton.geometry.get_coordinates().max()
palisades_max = palisades.geometry.get_coordinates().max()

# Max coords
max_x = [eaton_max.iloc[0], palisades_max.iloc[0]]
max_y = [eaton_max.iloc[1], palisades_max.iloc[1]]

# Plotting
fig, ax = plt.subplots(figsize=(15,10))

landsat[['swir22', 'nir08', 'red']].to_dataarray().plot.imshow(ax=ax, robust=True)
eaton.boundary.plot(ax=ax, color='blue', linewidth=2.5, label = 'Eaton Fire')
palisades.boundary.plot(ax=ax, color='yellow', linewidth=2.5, label = 'Palisades Fire')

plt.xlabel('x-Coordinate Projection (meter)')
plt.ylabel('y-Coordinate Projection (meter)')
plt.title('Eaton and Palisades Fire Perimeters in January 2021')
plt.legend()

# Add fire names as text annotations
combined_fires = pd.concat([eaton, palisades])
for x, y, name in zip(max_x, max_y, combined_fires['FIRE_NAME']):
    ax.text(x,y-9000,name, fontsize=16, ha='left', va = 'top', color= 'white', fontweight='bold')

plt.show()